In [18]:
pip install dotenv openai requests minsearch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 62.9 MB/s  0:00:00s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 59.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 60.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 63.3 MB/s  0:00:00s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [minsearch]━ 7/9 [scikit-learn]]

[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [78]:
def llm(prompt):
    response = openai_client.responses.create(
        model = "gpt-5.4-mini",
        input = prompt
    )
    return response.output_text

In [7]:
llm("Hey, what's up?")

'Hey! Not much—just here and ready to help. What’s on your mind?'

In [8]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely — in most cases, you can join after the course has started.

A few things to check:
- **Enrollment status:** Is the course still open for registration?
- **Missed material:** You may need to catch up on lectures, assignments, or announcements.
- **Access to recordings/resources:** See whether past sessions are available.
- **Deadlines:** Some courses have add/drop limits or late enrollment cutoffs.

If you want, I can help you draft a short message to the instructor asking whether late enrollment is allowed.


In [9]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [10]:
prompt = f"""
Your task is to answer questions from the course particpants based on the provided context.

Use the context to find relevant information and provide accurate answers. If the answer is ot found in the context, respond with "I do not know."

Question:
{question}

Context:
{context}
"""

In [11]:
answer = llm(prompt)
print(answer)

Yes, you can still join. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [13]:
import requests

doc_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(doc_url)
courses_raw = response.json()

In [14]:
print(courses_raw)

[{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 471}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 139}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 253}]


In [16]:
##course_url = "https://datatalks.club/faq/json/data-engineering-zoomcamp.json"
##course_response = requests.get(course_url)
##course_response.raise_for_status()
##coure_data = course_response.json()
##print(coure_data)

In [15]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""
    #print(course_url)

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)


1401

In [17]:
documents[33]

{'id': '1ba19ed6a0',
 'course': 'data-engineering-zoomcamp',
 'section': 'Environment & Setup',
 'question': 'Git Bash: Backslash as an escape character in Git Bash for Windows',
 'answer': 'For those who wish to use the backslash as an escape character in Git Bash for Windows, type the following in the terminal:\n\n```bash\nbash.escapeChar=\\\n```\n\n(Note: There is no need to include this in your `.bashrc` file.)'}

In [58]:
### Create our DB Index on the documents from the FAQ
from minsearch import Index

index = Index(
    text_fields = ["question", "section", "section"],
    keyword_fields = ["course"]
)

index.fit(documents)

In [70]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict = {"question": 2.0, "section": 0.5},
    filter_dict = {"course": "llm-zoomcamp"},
    num_results = 3
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '193612db63',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Orchestration',
  'question': "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
  'answer': "Notebooks are grea

In [60]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?"]

In [61]:
question = "Who is the course teacher?"
results = index.search(
    question,
    filter_dict = {"course": "llm-zoomcamp"}
)
[doc["question"] for doc in results]

['How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?',
 'I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'Why is the number of documents in the FAQ dataset different from the video, and why do my RAG results differ?',
 'Where is the LLM Zoomcamp Telegram channel?',
 'The homework submission form is still open even though the deadline has passed — can I still submit?',
 'Will the name I put in the certificate field be shown publicly online or shared with third parties?']

In [71]:
def search(question, course = "llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict = boost_dict,#
        filter_dict = filter_dict,
        num_results = 5
    )

In [72]:
search_results = search(question)
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?']

In [43]:
### Instructions

In [73]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers. 
If the answer is not found in the context, respond with "I don't know."
"""

In [74]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [75]:
### The context: a formatted string with all the search results (from the DB Index)
def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")
        
    return "\n".join(lines).strip()

In [55]:
#build_context(search_results)

In [76]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question = question,
        context = context,
    )
    return prompt.strip()

In [77]:
build_prompt(question, search_results)

'Question:\nI just discovered the course. Can I join now?\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nModule 3: Orchestration\nQ: Why do we need orchestration / Kestra — can\'t I just run the code in a notebook?\nA: Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, monitoring, secret management, and reliably chaining tasks together. That

In [79]:
response = openai_client.responses.create(
        model = "gpt-5.4-mini",
        input = prompt
    )

In [80]:
response.output[0]

ResponseOutputMessage(id='msg_0d05074aaedf68d4006a70ba586d348191b5d431ba6197d462', content=[ResponseOutputText(annotations=[], text='Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [81]:
response.output[0].content[0].text

'Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.'

In [82]:
response.output_text

'Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.'

In [83]:
response.usage

ResponseUsage(input_tokens=248, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=32, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=280)

In [84]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00033

In [86]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt},
]

response = openai_client.responses.create(
    model = "gpt-5.4-mini",
    input = message_history
)

In [93]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model = model,
        input = message_history
    )

    return response.output_text
    

In [94]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [95]:
answer = rag("I just discoveredthe course. Can I join now?")
print(answer)

Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.


In [98]:
answer = rag("How much student attended the course last year?")
print(answer)

I don't know.
